<a href="https://colab.research.google.com/github/pranav-358/amazon_ml_challenge/blob/main/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install rapidfuzz tqdm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

from rapidfuzz import fuzz
from catboost import CatBoostClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
BASE_PATH = Path("/content/Amazon/Train Dataset")

TRAIN_SOURCE1 = BASE_PATH / "train_source1.tsv"
TRAIN_SOURCE2 = BASE_PATH / "train_source2.tsv"
TRAIN_SOURCE3 = BASE_PATH / "train_source3.tsv"
TRAIN_GROUND_TRUTH = BASE_PATH / "train_ground_truth.tsv"

print(TRAIN_SOURCE1)

/content/Amazon/Train Dataset/train_source1.tsv


In [ ]:
BASE_PATH = Path("/content/Amazon/Train Dataset")

TRAIN_SOURCE1 = BASE_PATH / "train_source1.tsv"
TRAIN_SOURCE2 = BASE_PATH / "train_source2.tsv"
TRAIN_SOURCE3 = BASE_PATH / "train_source3.tsv"
TRAIN_GROUND_TRUTH = BASE_PATH / "train_ground_truth.tsv"

print(TRAIN_SOURCE1)

/content/Amazon/Train Dataset/train_source1.tsv


In [ ]:
train_source1 = pd.read_csv(TRAIN_SOURCE1, sep="\t")
train_source2 = pd.read_csv(TRAIN_SOURCE2, sep="\t")
train_source3 = pd.read_csv(TRAIN_SOURCE3, sep="\t")
train_ground_truth = pd.read_csv(TRAIN_GROUND_TRUTH, sep="\t")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [ ]:
def dataset_summary(df, name):
    print(f"\n{name}")
    print("-" * 40)
    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")
    print(f"Memory  : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
    print(f"Duplicate rows : {df.duplicated().sum():,}")

dataset_summary(train_source1, "Train Source 1")
dataset_summary(train_source2, "Train Source 2")
dataset_summary(train_source3, "Train Source 3")
dataset_summary(train_ground_truth, "Ground Truth")


Train Source 1
----------------------------------------
Rows    : 683,045
Columns : 4
Memory  : 187.07 MB
Duplicate rows : 0

Train Source 2
----------------------------------------
Rows    : 614,948
Columns : 4
Memory  : 173.31 MB
Duplicate rows : 0

Train Source 3
----------------------------------------
Rows    : 638,005
Columns : 4
Memory  : 178.49 MB
Duplicate rows : 0

Ground Truth
----------------------------------------
Rows    : 1,074,987
Columns : 2
Memory  : 156.44 MB
Duplicate rows : 0


In [ ]:
def dataset_summary(df, name):
    print(f"\n{name}")
    print("-" * 40)
    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")
    print(f"Memory  : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
    print(f"Duplicate rows : {df.duplicated().sum():,}")

dataset_summary(train_source1, "Train Source 1")
dataset_summary(train_source2, "Train Source 2")
dataset_summary(train_source3, "Train Source 3")
dataset_summary(train_ground_truth, "Ground Truth")


Train Source 1
----------------------------------------
Rows    : 683,045
Columns : 4
Memory  : 187.07 MB
Duplicate rows : 0

Train Source 2
----------------------------------------
Rows    : 614,948
Columns : 4
Memory  : 178.96 MB
Duplicate rows : 0

Train Source 3
----------------------------------------
Rows    : 638,005
Columns : 4
Memory  : 178.49 MB
Duplicate rows : 0

Ground Truth
----------------------------------------
Rows    : 1,074,987
Columns : 2
Memory  : 156.44 MB
Duplicate rows : 0


In [ ]:
print("Train Source 1")
display(train_source1.head(5))

print("Train Source 2")
display(train_source2.head(5))

print("Train Source 3")
display(train_source3.head(5))

print("Ground Truth")
display(train_ground_truth.head(5))

Train Source 1


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India


Train Source 2


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


Train Source 3


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block Jayanagar, Bengaluru Urban, Bangalore, ಕರ್ನಾಟಕ",India


Ground Truth


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785"


In [ ]:
missing_s1 = train_source1.isnull().sum().to_frame("Missing")
missing_s2 = train_source2.isnull().sum().to_frame("Missing")
missing_s3 = train_source3.isnull().sum().to_frame("Missing")
missing_gt = train_ground_truth.isnull().sum().to_frame("Missing")

print("Source 1")
display(missing_s1)

print("Source 2")
display(missing_s2)

print("Source 3")
display(missing_s3)

print("Ground Truth")
display(missing_gt)

Source 1


,Missing
entity_id,0
business_name,1
business_address,1
country,1


Source 2


,Missing
entity_id,0
business_name,0
business_address,20391
country,1


Source 3


,Missing
entity_id,0
business_name,1
business_address,21068
country,1


Ground Truth


,Missing
source1_entity_id,0
matched_entity_ids,60340


In [ ]:
print("Source 1")
display(train_source1.dtypes)

print("Source 2")
display(train_source2.dtypes)

print("Source 3")
display(train_source3.dtypes)

print("Ground Truth")
display(train_ground_truth.dtypes)

Source 1


,0
entity_id,object
business_name,object
business_address,object
country,object


Source 2


,0
entity_id,object
business_name,object
business_address,object
country,object


Source 3


,0
entity_id,object
business_name,object
business_address,object
country,object


Ground Truth


,0
source1_entity_id,object
matched_entity_ids,object


In [ ]:
country_dist = train_source1["country"].value_counts().reset_index()
country_dist.columns = ["Country", "Businesses"]

display(country_dist)

,Country,Businesses
0,US,409504
1,India,273540


In [ ]:
print("Unique business names :", train_source1["business_name"].nunique())
print("Unique addresses      :", train_source1["business_address"].nunique())
print("Unique countries      :", train_source1["country"].nunique())

Unique business names : 536160
Unique addresses      : 671845
Unique countries      : 2
